In [1]:
import os
from dotenv import load_dotenv

# Load variables from the .env file
load_dotenv()

# Read the API key
api_key = os.getenv("API_KEY")

if api_key is None:
    raise ValueError("API_KEY is not set!")

print("API key loaded successfully:", bool(api_key))

API key loaded successfully: True


In [2]:
from openai import OpenAI

client = OpenAI(base_url="https://api.gapgpt.app/v1", api_key=api_key)

response = client.chat.completions.create(
    model="gapgpt-gemma-26b-a4b",
    messages=[{"role": "user", "content": "سلام!"}]
)
print(response.choices[0].message.content)

سلام! چطور می‌توانم به شما کمک کنم؟


In [4]:
response = client.responses.create(
    model="gapgpt-gemma-26b-a4b",
    input="سلام!"
)
print(response.output_text)

سلام! چطور می‌توانم به شما کمک کنم؟ :)


In [5]:
demos = [
    ("This movie was amazing and inspiring.", "Positive"),
    ("I really enjoyed this product.", "Positive"),
    ("The acting was terrible and boring.", "Negative"),
    ("This product broke after one day.", "Negative"),
    ("The film was beautiful and entertaining.", "Positive"),
    ("The quality was poor and disappointing.", "Negative"),
]

tests = [
    ("I loved the story and the performances.", "Positive"),
    ("This is the worst product I have ever bought.", "Negative"),
    ("The movie was surprisingly good.", "Positive"),
    ("It stopped working after a few hours.", "Negative"),
]

MODEL = "gapgpt-gemma-26b-a4b"

def classify_sentiment(sentence, examples=None):
    prompt = """
        Classify the sentiment of the given sentence.
        Return exactly one word: Positive or Negative.
    """

    if examples is not None:
        prompt += "\nExamples:\n"
        for text, label in examples:
            prompt += f'Sentence: "{text}"\nSentiment: {label}\n'

    prompt += f'\nSentence: "{sentence}"\nSentiment:'

    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=0,
    )

    return response.output_text.strip()

In [6]:
# 1. Zero-shot: no labeled examples
# 2. Balanced few-shot: 2 positive + 2 negative
balanced = demos[:4]

# 3. Imbalanced few-shot: 3 positive + 1 negative
imbalanced = [
    demos[0], demos[1], demos[4], demos[2]
]

# 4. Reverse the order of the same imbalanced examples
reversed_imbalanced = list(reversed(imbalanced))

experiments = {
    "Zero-shot": None,
    "Balanced few-shot": balanced,
    "Imbalanced few-shot (P P P N)": imbalanced,
    "Reversed few-shot (N P P P)": reversed_imbalanced,
}

for experiment_name, examples in experiments.items():
    print(f"\n--- {experiment_name} ---")

    for sentence, true_label in tests:
        prediction = classify_sentiment(sentence, examples)
        correct = prediction.lower() == true_label.lower()

        print(f"Sentence: {sentence}")
        print(f"True: {true_label} | Predicted: {prediction}")
        print(f"Correct: {correct}\n")


--- Zero-shot ---
Sentence: I loved the story and the performances.
True: Positive | Predicted: Positive
Correct: True

Sentence: This is the worst product I have ever bought.
True: Negative | Predicted: Negative
Correct: True

Sentence: The movie was surprisingly good.
True: Positive | Predicted: Positive
Correct: True

Sentence: It stopped working after a few hours.
True: Negative | Predicted: Negative
Correct: True


--- Balanced few-shot ---
Sentence: I loved the story and the performances.
True: Positive | Predicted: Positive
Correct: True

Sentence: This is the worst product I have ever bought.
True: Negative | Predicted: Negative
Correct: True

Sentence: The movie was surprisingly good.
True: Positive | Predicted: Positive
Correct: True

Sentence: It stopped working after a few hours.
True: Negative | Predicted: Negative
Correct: True


--- Imbalanced few-shot (P P P N) ---
Sentence: I loved the story and the performances.
True: Positive | Predicted: Positive
Correct: True

Sen

### نتایج آزمایش

| روش                               | پیش‌بینی صحیح | دقت  |
| --------------------------------- | ------------- | ---- |
| Zero-shot                         | ۴ از ۴        | ۱۰۰٪ |
| Few-shot متعادل                   | ۴ از ۴        | ۱۰۰٪ |
| Few-shot نامتوازن (P P P N)       | ۴ از ۴        | ۱۰۰٪ |
| Few-shot با ترتیب معکوس (N P P P) | ۴ از ۴        | ۱۰۰٪ |

### پاراگراف تحلیل برای گزارش

در این آزمایش، اثر قابل‌مشاهده‌ای از Majority Label Bias یا Recency Bias مشاهده نشد؛ زیرا مدل در هر چهار روش، تمام جمله‌های تست را به‌درستی طبقه‌بندی کرد. با وجود تغییر توزیع و ترتیب مثال‌های آموزشی، پیش‌بینی‌ها بدون تغییر باقی ماندند. یکی از دلایل احتمالی این نتیجه، توانایی مدل مکالمه‌محور (Chat-tuned) در درک معنای جمله‌ها و پیروی از دستورالعمل طبقه‌بندی احساسات است. همچنین، کوچک بودن مجموعه تست (فقط چهار جمله) و واضح بودن احساسات آن‌ها ممکن است مانع آشکار شدن این سوگیری‌ها شده باشد. بنابراین، نمی‌توان نتیجه گرفت که مدل اساساً فاقد این سوگیری‌هاست؛ برای بررسی دقیق‌تر، به مجموعه تست بزرگ‌تر و جمله‌های مبهم‌تر نیاز داریم.

In [16]:
math_problems = [
    (
        "Ali has 3 boxes of eggs. Each box contains 12 eggs. "
        "He uses 8 eggs and buys 5 more. How many eggs does he have now?",
        "33"
    ),
    (
        "Sara buys 4 notebooks for $3 each and 2 pens for $2 each. "
        "How much does she pay in total?",
        "16"
    ),
    (
        "A bus has 45 passengers. At the next stop, 12 passengers get off "
        "and 7 passengers get on. How many passengers are on the bus now?",
        "40"
    ),
    (
        "A farmer has 6 baskets with 8 apples in each basket. "
        "He sells 17 apples. How many apples are left?",
        "31"
    ),
    (
        "Reza has $50. He buys 3 books for $8 each. "
        "How much money does he have left?",
        "26"
    ),
]

MODEL = "gpt-5.5"

def solve_question(question):
    prompt = f"""
        Solve the following math word problem.

        Question: {question}

        Return only the final numerical answer.
        Do not include any explanation or extra text.
    """

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    return response.output_text.strip()

In [17]:
# Run the questions and compare predictions with correct answers
results = []

for question, correct_answer in math_problems:
    predicted_answer = solve_question(question)

    # Normalize the answers for comparison
    predicted = predicted_answer.replace(",", "").strip()
    expected = correct_answer.replace(",", "").strip()

    is_correct = predicted == expected

    results.append({
        "question": question,
        "correct_answer": correct_answer,
        "predicted_answer": predicted_answer,
        "is_correct": is_correct,
    })

    print(f"Question: {question}")
    print(f"Correct answer: {correct_answer}")
    print(f"Predicted answer: {predicted_answer}")
    print(f"Correct prediction: {is_correct}")
    print("-" * 60)

# Calculate accuracy
correct_count = sum(result["is_correct"] for result in results)
accuracy = correct_count / len(results) * 100

print(f"\nCorrect predictions: {correct_count}/{len(results)}")
print(f"Accuracy: {accuracy:.1f}%")

Question: Ali has 3 boxes of eggs. Each box contains 12 eggs. He uses 8 eggs and buys 5 more. How many eggs does he have now?
Correct answer: 33
Predicted answer: 33
Correct prediction: True
------------------------------------------------------------
Question: Sara buys 4 notebooks for $3 each and 2 pens for $2 each. How much does she pay in total?
Correct answer: 16
Predicted answer: 16
Correct prediction: True
------------------------------------------------------------
Question: A bus has 45 passengers. At the next stop, 12 passengers get off and 7 passengers get on. How many passengers are on the bus now?
Correct answer: 40
Predicted answer: 40
Correct prediction: True
------------------------------------------------------------
Question: A farmer has 6 baskets with 8 apples in each basket. He sells 17 apples. How many apples are left?
Correct answer: 31
Predicted answer: 31
Correct prediction: True
------------------------------------------------------------
Question: Reza has $5

In [18]:
def solve_question_cot(question):
    prompt = f"""
        Let's think step by step.

        Solve the following math word problem carefully.

        Question: {question}

        After solving the problem, return only the final numerical answer.
        Do not include any explanation in your final answer.
    """

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    return response.output_text.strip()


# Run all questions and compare predictions with correct answers
results = []

for question, correct_answer in math_problems:
    predicted_answer = solve_question_cot(question)

    predicted = predicted_answer.replace(",", "").strip()
    expected = correct_answer.replace(",", "").strip()

    is_correct = predicted == expected

    results.append({
        "question": question,
        "correct_answer": correct_answer,
        "predicted_answer": predicted_answer,
        "is_correct": is_correct,
    })

    print(f"Question: {question}")
    print(f"Correct answer: {correct_answer}")
    print(f"Predicted answer: {predicted_answer}")
    print(f"Correct prediction: {is_correct}")
    print("-" * 60)


# Calculate accuracy
correct_count = sum(r["is_correct"] for r in results)
total_count = len(results)
accuracy = correct_count / total_count * 100 if total_count else 0

print(f"\nCorrect predictions: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.1f}%")


Question: Ali has 3 boxes of eggs. Each box contains 12 eggs. He uses 8 eggs and buys 5 more. How many eggs does he have now?
Correct answer: 33
Predicted answer: 33
Correct prediction: True
------------------------------------------------------------
Question: Sara buys 4 notebooks for $3 each and 2 pens for $2 each. How much does she pay in total?
Correct answer: 16
Predicted answer: 16
Correct prediction: True
------------------------------------------------------------
Question: A bus has 45 passengers. At the next stop, 12 passengers get off and 7 passengers get on. How many passengers are on the bus now?
Correct answer: 40
Predicted answer: 40
Correct prediction: True
------------------------------------------------------------
Question: A farmer has 6 baskets with 8 apples in each basket. He sells 17 apples. How many apples are left?
Correct answer: 31
Predicted answer: 31
Correct prediction: True
------------------------------------------------------------
Question: Reza has $5

In [19]:
import re
from collections import Counter

def solve_question(question):
    prompt = f"""
        Let's think step by step.

        Solve the following math word problem carefully.

        Question: {question}

        Return only the final numerical answer.
        Do not include any explanation or extra text.
    """

    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=0.7,
    )

    return response.output_text.strip()


def extract_number(answer):
    """Extract a numeric answer from the model output."""
    cleaned = answer.replace(",", "").strip()
    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)

    if match:
        return match.group()

    return None


N_RUNS = 5
results = []

for question, correct_answer in math_problems:
    predictions = []

    # Generate five independent answers
    for run in range(N_RUNS):
        raw_answer = solve_question(question)
        number = extract_number(raw_answer)
        predictions.append(number)

        print(f"Run {run + 1}: {raw_answer}")

    # Majority vote, ignoring responses with no extractable number
    valid_predictions = [p for p in predictions if p is not None]
    counts = Counter(valid_predictions)

    if counts:
        highest_votes = max(counts.values())
        winners = [
            answer for answer, votes in counts.items()
            if votes == highest_votes
        ]

        # If tied, select the first answer to reach the highest vote count
        if len(winners) == 1:
            final_answer = winners[0]
        else:
            final_answer = next(
                p for p in valid_predictions if p in winners
            )
    else:
        final_answer = None

    is_correct = final_answer == correct_answer

    results.append({
        "question": question,
        "correct_answer": correct_answer,
        "predictions": predictions,
        "vote_counts": dict(counts),
        "final_answer": final_answer,
        "is_correct": is_correct,
    })

    print(f"Votes: {dict(counts)}")
    print(f"Majority answer: {final_answer}")
    print(f"Correct answer: {correct_answer}")
    print(f"Correct prediction: {is_correct}")
    print("-" * 60)


# Calculate overall accuracy
correct_count = sum(r["is_correct"] for r in results)
total_count = len(results)
accuracy = correct_count / total_count * 100 if total_count else 0

print(f"\nCorrect predictions: {correct_count}/{total_count}")
print(f"Self-Consistency accuracy: {accuracy:.1f}%")


Run 1: 33
Run 2: 33
Run 3: 33
Run 4: 33
Run 5: 33
Votes: {'33': 5}
Majority answer: 33
Correct answer: 33
Correct prediction: True
------------------------------------------------------------
Run 1: 16
Run 2: 16
Run 3: 16
Run 4: 16
Run 5: 16
Votes: {'16': 5}
Majority answer: 16
Correct answer: 16
Correct prediction: True
------------------------------------------------------------
Run 1: 40
Run 2: 40
Run 3: 40
Run 4: 40
Run 5: 40
Votes: {'40': 5}
Majority answer: 40
Correct answer: 40
Correct prediction: True
------------------------------------------------------------
Run 1: 31
Run 2: 31
Run 3: 31
Run 4: 31
Run 5: 31
Votes: {'31': 5}
Majority answer: 31
Correct answer: 31
Correct prediction: True
------------------------------------------------------------
Run 1: 26
Run 2: 26
Run 3: 26
Run 4: 26
Run 5: 26
Votes: {'26': 5}
Majority answer: 26
Correct answer: 26
Correct prediction: True
------------------------------------------------------------

Correct predictions: 5/5
Self-Consisten

## جدول مقایسهٔ دقت و هزینهٔ سه روش

| روش                       | تعداد فراخوانی API | پاسخ صحیح | دقت  | هزینهٔ نسبی           |
| ------------------------- | ------------------ | --------- | ---- | --------------------- |
| A: Direct Prompting       | ۵                  | ۵ از ۵    | ۱۰۰٪ | کم                    |
| B: Chain-of-Thought (CoT) | ۵                  | ۵ از ۵    | ۱۰۰٪ | کم تا متوسط           |
| C: Self-Consistency       | ۲۵                 | ۵ از ۵    | ۱۰۰٪ | حدود ۵ برابر یا بیشتر |

هزینهٔ نسبی تقریبی است؛ هزینهٔ واقعی به تعداد توکن‌های ورودی و خروجی و قیمت مدل بستگی دارد. در روش C، برای هر یک از ۵ سؤال، مدل ۵ بار فراخوانی شده است.

## تحلیل کوتاه

در این آزمایش، روش Self-Consistency با وجود افزایش تعداد فراخوانی‌های API از ۵ به ۲۵، دقت را نسبت به دو روش دیگر افزایش نداد؛ زیرا هر سه روش به دقت ۱۰۰٪ رسیدند. بنابراین، برای این مجموعهٔ کوچک و ساده از مسائل، افزایش هزینه در روش C ارزشش را نداشت؛ هرچند در مسائل پیچیده‌تر که پاسخ‌های مدل متغیرند، رأی‌گیری بین چند پاسخ ممکن است دقت را بهبود دهد.


In [31]:
import os, json, time, textwrap, requests

WIKI = "https://en.wikipedia.org/w/api.php"
CACHE_FILE = "rag-wiki-cache.json"
CACHE = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def show(label, text):
    print(f"── {label} " + "─" * max(0, 60 - len(label)))
    if text is None: 
        print("\nNone\n")
    else:
     print("\n".join(textwrap.fill(line, 100) for line in text.split("\n")))

def page(title):
    # plain text of one Wikipedia page (cached)
    params = dict(action="query", prop="extracts", explaintext=1, titles=title, redirects=1)
    key = json.dumps(params, sort_keys=True)
    if key not in CACHE:
        r = requests.get(WIKI, params={"format": "json", **params},
                         headers={"User-Agent": "week15-rag-handson/0.1"}, timeout=20)
        CACHE[key] = r.json()
        json.dump(CACHE, open(CACHE_FILE, "w"))
    return next(iter(CACHE[key]["query"]["pages"].values())).get("extract", "")

REFERENCE = page("Napoleon").split("\n\n\n")[0]   # the lead, before the first "== Section =="
show("reference", REFERENCE)

── reference ───────────────────────────────────────────────────
Napoleon Bonaparte (born Napoleone di Buonaparte; 15 August 1769 – 5 May 1821), later known by his
regnal name Napoleon I, was Emperor of the French from 18 May 1804 until his first abdication in
1814, with a brief restoration during the Hundred Days in 1815. He rose to prominence as a brilliant
general during the French Revolution and led a series of military campaigns across Europe and the
Middle East during the French Revolutionary and Napoleonic Wars. As a statesman, he implemented
numerous legal and administrative reforms in France and Europe.
Born on the island of Corsica to a family of Italian origin, Napoleon moved to mainland France in
1779 and was commissioned as an officer in the French Royal Army in 1785. He supported the French
Revolution in 1789 and promoted its cause in Corsica. He rose rapidly through the ranks after
winning the siege of Toulon in 1793 and defeating royalist insurgents in Paris on 13 Vendé

In [21]:
def chat(messages, model=MODEL, client=client, **kw):
    # /v1/chat/completions: send messages, get one assistant turn.
    kw = {"max_tokens": 120, "temperature": 0, **kw}
    try:
        r = client.chat.completions.create(model=model, messages=messages, **kw)
        return r.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e}"

In [29]:
GROUNDED = ("Answer using only the reference text below. Quote the sentence you used. "
            "If the reference does not contain the answer, reply exactly: Not in the reference.")

QUESTIONS = [
    "Where and when was Napoleon Bonaparte born?",                      # in the reference
    "What is the name of a man who killed Napoleon?",                   # not in the reference, and the model can't know it
    "What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar??",     # not in the reference, but the model knows it
    "ٌIn what year did Napoleon win the Battle of Waterloo??"
]

In [25]:
def ask(question, reference=REFERENCE, system=None):
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": f"Reference:\n{reference}\n\nQuestion: {question}"})
    return chat(messages)

In [32]:
for q in QUESTIONS:
    show(f"pasted only · {q}", ask(q))
    show(f"grounded    · {q}", ask(q, system=GROUNDED))
    print()

── pasted only · Where and when was Napoleon Bonaparte born? ───
Napoleon Bonaparte was born on the island of Corsica on **15 August 1769**.
── grounded    · Where and when was Napoleon Bonaparte born? ───
Napoleon Bonaparte was born on the island of Corsica on 15 August 1769.

Quotes used: “Napoleon Bonaparte (born Nap

── pasted only · What is the name of a man who killed Napoleon? 
No man killed Napoleon. According to the reference, Napoleon died of stomach cancer on Saint Helena
in 1821.
── grounded    · What is the name of a man who killed Napoleon? 
Not in the reference.

── pasted only · What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar?? 
The British admiral was **Horatio Nelson** (Admiral Lord Nelson).
── grounded    · What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar?? 
Not in the reference.

── pasted only · ٌIn what year did Napoleon win the Battle of Waterloo?? 
Napoleon did **n

In [38]:
NEW_REFERENCE = REFERENCE[78:]
show("new reference", NEW_REFERENCE)

── new reference ───────────────────────────────────────────────
, later known by his regnal name Napoleon I, was Emperor of the French from 18 May 1804 until his
first abdication in 1814, with a brief restoration during the Hundred Days in 1815. He rose to
prominence as a brilliant general during the French Revolution and led a series of military
campaigns across Europe and the Middle East during the French Revolutionary and Napoleonic Wars. As
a statesman, he implemented numerous legal and administrative reforms in France and Europe.
Born on the island of Corsica to a family of Italian origin, Napoleon moved to mainland France in
1779 and was commissioned as an officer in the French Royal Army in 1785. He supported the French
Revolution in 1789 and promoted its cause in Corsica. He rose rapidly through the ranks after
winning the siege of Toulon in 1793 and defeating royalist insurgents in Paris on 13 Vendémiaire in
1795. In 1796 he commanded a military campaign against the Austrians

In [40]:
for q in QUESTIONS:
    show(f"pasted only · {q}", ask(q, reference=NEW_REFERENCE))
    show(f"grounded    · {q}", ask(q, system=GROUNDED, reference=NEW_REFERENCE))
    print()

── pasted only · Where and when was Napoleon Bonaparte born? ───

None

── grounded    · Where and when was Napoleon Bonaparte born? ───
Not in the reference.

── pasted only · What is the name of a man who killed Napoleon? 
No man killed Napoleon. According to the reference, Napoleon died of stomach cancer in 1821 on Saint
Helena.
── grounded    · What is the name of a man who killed Napoleon? 
Not in the reference.

── pasted only · What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar?? 
Admiral **Horatio Nelson** — also known as **Lord Nelson** — defeated Napoleon’s fleet at the
**Battle of Trafalgar**.
── grounded    · What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar?? 
Not in the reference.

── pasted only · ٌIn what year did Napoleon win the Battle of Waterloo?? 
Napoleon did **not** win the Battle of Waterloo; he was **defeated** there in **1815**.
── grounded    · ٌIn what year did Nap

## گزارش آزمایش Grounded Prompting

### جدول نتایج بخش ۳ و ۴

در این آزمایش، پاسخ مدل در دو حالت مقایسه شده است: حالت عادی (`pasted only`) و حالت مقید به متن مرجع (`grounded`). در روش Grounded، مدل باید فقط از متن مرجع استفاده کند و در صورت نبود پاسخ، دقیقاً عبارت `Not in the reference` را برگرداند.

| سؤال                                                                                               | نوع سؤال                                         | حالت عادی (بدون قوانین)                                          | حالت Grounded (با قوانین)                                            |
| -------------------------------------------------------------------------------------------------- | ------------------------------------------------ | ---------------------------------------------------------------- | -------------------------------------------------------------------- |
| Where and when was Napoleon Bonaparte born?                                                        | پاسخ در متن هست و خروجی دوم پاسخ از متن پاک شده است                                  | محل و تاریخ تولد را درست بیان کرد؛ در اجرای دوم `None` برگرداند. | در اجرای اول پاسخ صحیح داد؛ در اجرای دوم `Not in the reference` گفت. |
| What is the name of a man who killed Napoleon?                                                     | پیش‌فرض غلط / اطلاعات ناموجود                    | پیش‌فرض را اصلاح کرد و گفت کسی ناپلئون را نکشت.                  | در هر دو اجرا `Not in the reference` گفت.                            |
| What was the name of the British admiral who defeated Napoleon's fleet at the Battle of Trafalgar? | پاسخ در دانش عمومی مدل هست، اما در متن مرجع نیست | نام Horatio Nelson را در هر دو اجرا بیان کرد.                    | در هر دو اجرا `Not in the reference` گفت.                            |
| In what year did Napoleon win the Battle of Waterloo?                                              | پیش‌فرض غلط                                      | پیش‌فرض غلط را اصلاح کرد و گفت ناپلئون در سال ۱۸۱۵ شکست خورد.    | در اجرای اول `Not in the reference` و در اجرای دوم `None` برگرداند.  |

توضیح: در این جدول، `None` به معنای خروجی خالی یا نبود پاسخ قابل‌استفاده در اجرای ثبت‌شده است؛ نباید آن را معادل پاسخ صحیح یا عبارت مورد انتظار `Not in the reference` در نظر گرفت.


In [41]:
from enum import Enum

from pydantic import BaseModel, ValidationError, ConfigDict

class Sentiment(str, Enum):
    positive = "positive"
    negative = "negative"
    neutral = "neutral"


class SentimentResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    reasoning: str
    label: Sentiment


reviews = [
    "This movie was amazing and inspiring.",
    "The acting was terrible and boring.",
    "The product is okay, nothing special.",
    "Not worth the ticket.",
    "I expected it to be bad, but it was actually wonderful.",
]


def make_prompt(review, key_order="reasoning_first"):
    if key_order == "reasoning_first":
        format_example = (
            '{"reasoning": "Short explanation", '
            '"label": "positive"}'
        )
    else:
        format_example = (
            '{"label": "positive", '
            '"reasoning": "Short explanation"}'
        )

    return f"""
        Analyze the sentiment of this review.

        Review: {review}

        Choose exactly one label:
        positive, negative, or neutral.

        Return JSON with these two keys:
        - reasoning: a short explanation
        - label: the sentiment label

        Use this key order in your JSON example:
        {format_example}

        Return only the JSON object.
        """


def classify_review(
    review,
    method="prompt",
    key_order="reasoning_first",
):
    prompt = make_prompt(review, key_order)

    kwargs = {
        "model": MODEL,
        "input": prompt,
    }

    if method == "json_mode":
        kwargs["text"] = {
            "format": {"type": "json_object"}
        }

    elif method == "json_schema":
        schema = SentimentResult.model_json_schema()

        # Enforce the requested key order in the schema.
        properties = {}
        if key_order == "reasoning_first":
            properties["reasoning"] = {"type": "string"}
            properties["label"] = {
                "type": "string",
                "enum": ["positive", "negative", "neutral"],
            }
        else:
            properties["label"] = {
                "type": "string",
                "enum": ["positive", "negative", "neutral"],
            }
            properties["reasoning"] = {"type": "string"}

        schema = {
            "type": "object",
            "properties": properties,
            "required": ["reasoning", "label"],
            "additionalProperties": False,
        }

        kwargs["text"] = {
            "format": {
                "type": "json_schema",
                "name": "sentiment_result",
                "strict": True,
                "schema": schema,
            }
        }

    response = client.responses.create(**kwargs)
    raw_output = response.output_text

    print(f"\nMethod: {method}")
    print(f"Review: {review}")
    print(f"Raw output: {raw_output}")

    # 3. Validate the response using Pydantic
    try:
        parsed_json = json.loads(raw_output)
        validated = SentimentResult.model_validate(parsed_json)
        print("Pydantic validation: PASSED")
        print("Label:", validated.label.value)
        print("Reasoning:", validated.reasoning)
        return validated

    except (json.JSONDecodeError, ValidationError) as exc:
        print("Pydantic validation: FAILED")
        print(exc)
        return None


for review in reviews:
    for method in ["prompt", "json_mode", "json_schema"]:
        classify_review(review, method)


test_review = "Not worth the ticket."

for method in ["prompt", "json_mode", "json_schema"]:
    print(f"\n--- Key-order experiment: {method} ---")

    first = classify_review(
        test_review, method, "reasoning_first"
    )
    second = classify_review(
        test_review, method, "label_first"
    )

    if first and second:
        print("First label:", first.label.value)
        print("Second label:", second.label.value)
        print("Labels unchanged:", first.label == second.label)



Method: prompt
Review: This movie was amazing and inspiring.
Raw output: {"reasoning":"The review uses strongly favorable words like \"amazing\" and \"inspiring.\"","label":"positive"}
Pydantic validation: PASSED
Label: positive
Reasoning: The review uses strongly favorable words like "amazing" and "inspiring."

Method: json_mode
Review: This movie was amazing and inspiring.
Raw output: {"reasoning":"The review uses strongly favorable words like \"amazing\" and \"inspiring.\"","label":"positive"}
Pydantic validation: PASSED
Label: positive
Reasoning: The review uses strongly favorable words like "amazing" and "inspiring."

Method: json_schema
Review: This movie was amazing and inspiring.
Raw output: {"reasoning":"The review uses strongly favorable words like \"amazing\" and \"inspiring,\" indicating a positive sentiment.","label":"positive"}
Pydantic validation: PASSED
Label: positive
Reasoning: The review uses strongly favorable words like "amazing" and "inspiring," indicating a posi

In [ ]:
import ast
import operator

EXCHANGE_RATES = {
    "USD": 100_000,  # 1 USD = 100,000 Toman
}


def get_exchange_rate(currency: str) -> float:
    """Return the exchange rate for a currency in Toman."""
    currency = currency.upper()

    if currency not in EXCHANGE_RATES:
        raise ValueError(f"Unsupported currency: {currency}")

    return EXCHANGE_RATES[currency]


def calculator(expression: str) -> float:
    """Safely evaluate basic arithmetic expressions."""

    # Allow only basic arithmetic operations.
    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos,
    }

    def evaluate(node):
        if isinstance(node, ast.Expression):
            return evaluate(node.body)

        if isinstance(node, ast.Constant):
            if type(node.value) in (int, float):
                return node.value
            raise ValueError("Only numeric values are allowed.")

        if isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in allowed_operators:
                raise ValueError("Unsupported arithmetic operator.")
            return allowed_operators[op_type](
                evaluate(node.left),
                evaluate(node.right),
            )

        if isinstance(node, ast.UnaryOp):
            op_type = type(node.op)
            if op_type not in allowed_operators:
                raise ValueError("Unsupported unary operator.")
            return allowed_operators[op_type](evaluate(node.operand))

        raise ValueError("Invalid arithmetic expression.")

    tree = ast.parse(expression, mode="eval")
    return evaluate(tree)


tools = [
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get the fixed exchange rate of a currency in Toman.",
        "parameters": {
            "type": "object",
            "properties": {
                "currency": {
                    "type": "string",
                    "description": "Currency code, such as USD",
                }
            },
            "required": ["currency"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Arithmetic expression, such as 50 * 100000",
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

available_functions = {
    "get_exchange_rate": get_exchange_rate,
    "calculator": calculator,
}

def ask_with_tools(question: str, max_steps: int = 4):
    # Keep the entire conversation locally.
    conversation = [
        {"role": "user", "content": question}
    ]

    response = client.responses.create(
        model=MODEL,
        input=conversation,
        tools=tools,
    )

    for step in range(max_steps):
        # Save the model's response items in the conversation.
        conversation.extend(response.output)

        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        # If the model did not request any tools, return its answer.
        if not tool_calls:
            print(f"Finished after {step} tool-calling steps.")
            return response.output_text

        tool_outputs = []

        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.arguments)
            except (json.JSONDecodeError, TypeError) as exc:
                result = {"error": f"Invalid JSON: {exc}"}
            else:
                try:
                    if tool_call.name not in available_functions:
                        raise ValueError(
                            f"Unknown tool: {tool_call.name}"
                        )

                    function = available_functions[tool_call.name]
                    result = function(**arguments)

                except Exception as exc:
                    result = {"error": str(exc)}

            print(f"\nTool: {tool_call.name}")
            print(f"Arguments: {tool_call.arguments}")
            print(f"Result: {result}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(
                    result,
                    ensure_ascii=False
                ),
            })

        conversation.extend(tool_outputs)

        response = client.responses.create(
            model="gpt-4.1",
            input=conversation,
            tools=tools,
        )

    if not any(
        item.type == "function_call"
        for item in response.output
    ):
        return response.output_text

    return "Maximum tool-calling steps reached."


question = (
    "فرض کن نرخ هر دلار 100000 تومان است. "
    "۵۰ دلار چند تومان می‌شود و ۱۰ درصد این مبلغ چقدر است؟ "
    "برای محاسبات از ابزارها استفاده کن."
)

answer = ask_with_tools(question, max_steps=4)

print("\nFinal answer:")
print(answer)


Tool: calculator
Arguments: {"expression":"50 * 100000"}
Result: 5000000

Tool: calculator
Arguments: {"expression":"50 * 100000 * 0.10"}
Result: 500000.0
Finished after 1 tool-calling steps.

Final answer:
۵۰ دلار با نرخ هر دلار ۱۰۰,۰۰۰ تومان برابر با ۵,۰۰۰,۰۰۰ تومان می‌شود.
۱۰ درصد این مبلغ برابر با ۵۰۰,۰۰۰ تومان است.


## گزارش تمرین

### ۱. تفاوت «فرمت معتبر» و «پاسخ درست»

در آزمایش تحلیل احساسات، هر سه روش `prompt`، `json_mode` و `json_schema` برای پنج نظر، خروجی‌هایی تولید کردند که اعتبارسنجی Pydantic را با موفقیت پشت سر گذاشتند و برچسب‌های مناسبی داشتند. آزمایش تغییر ترتیب کلیدهای JSON نیز نشان داد که در این نمونه‌ها، جابه‌جایی کلیدهای `label` و `reasoning` نتیجهٔ طبقه‌بندی را تغییر نداد. بااین‌حال، معتبر بودن فرمت به معنای درست بودن پاسخ نیست؛ ممکن است خروجی از نظر ساختار کاملاً معتبر باشد، اما برچسب اشتباهی داشته باشد. بنابراین، علاوه بر اعتبارسنجی ساختاری، باید درستی معنایی پاسخ را نیز با داده‌های آزمون یا معیارهای ارزیابی بررسی کرد.

### ۲. چرا برنامه باید مجوز اجرای ابزار را کنترل کند؟

در آزمایش Tool Calling، مدل درخواست اجرای ابزار `calculator` را برای محاسبهٔ مبلغ ۵۰ دلار و سپس ۱۰ درصد آن تولید کرد. برنامه درخواست‌ها را دریافت کرد، تابع محلی را اجرا کرد و نتایج را برای تولید پاسخ نهایی در اختیار مدل گذاشت. مدل صرفاً پیشنهاددهندهٔ اجرای ابزار است و نباید مستقیماً اختیار اجرای توابع محلی را داشته باشد؛ زیرا اجرای خودکار و بدون کنترل درخواست‌ها می‌تواند به عملیات ناخواسته، دسترسی غیرمجاز یا اجرای ورودی‌های خطرناک منجر شود. به همین دلیل، برنامه باید نام ابزار، آرگومان‌ها و مجاز بودن عملیات را بررسی کند و تنها پس از تأیید آن‌ها، تابع مربوطه را اجرا کند.